In [1]:
# Cell R1: TerraTorch installation on the CPU runtime (restart the kernel after this cell)
!pip install -q terratorch rasterio


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 656.6/656.6 kB 14.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 859.3/859.3 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.8/174.8 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 83.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 688.1/688.1 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.4/72.4 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.6/165.6 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.8/168.8 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━

In [1]:
# Cell R2: Drive mount, repository and import path
import os
import sys
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

REPO_DIR = Path("/content/floodfm-sen1floods11")
if REPO_DIR.exists():
    !git -C {REPO_DIR} pull
else:
    !git clone https://github.com/Prhm93/floodfm-sen1floods11.git {REPO_DIR}
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

CONTENT = Path("/content/drive/MyDrive/2026Research/FloodFM-Sen1Floods11/content")
DATA_ROOT = CONTENT / "data" / "sen1floods11"
CKPT_ROOT = CONTENT / "checkpoints"
RESULTS_DIR = CONTENT / "results"
PRED_DIR = CONTENT / "predictions"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cloning into '/content/floodfm-sen1floods11'...
remote: Enumerating objects: 104, done.
remote: Counting objects: 100% (104/104), done.
remote: Compressing objects: 100% (63/63), done.
remote: Total 104 (delta 50), reused 83 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (104/104), 4.94 MiB | 12.78 MiB/s, done.
Resolving deltas: 100% (50/50), done.


In [2]:
# Cell R3: CPU evaluation of the saved best checkpoint of prithvi_frozen_frac10_seed1
import json
import subprocess
import time
from datetime import datetime
import numpy as np
import torch
from floodfm.data import read_chip
from floodfm.metrics import confusion_counts, add_counts, scores
from floodfm.prithvi import build_prithvi, PrithviSen1Floods11, logits_of, EVAL_SPLITS

METHOD = "prithvi_frozen_frac10_seed1"
torch.set_num_threads(os.cpu_count())

# Best checkpoint saved on Drive before the Drive sync stopped (expected: epoch 56, valid IoU 0.7658)
state = torch.load(CKPT_ROOT / METHOD / "best.pt", map_location="cpu", weights_only=False)
print(f"Best checkpoint: epoch {state['epoch']}, valid IoU {state['valid_iou']:.4f}")

model = build_prithvi(freeze_backbone=True)
model.load_state_dict(state["model"])
model.eval()

commit = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
results = {
    "method": METHOD,
    "description": ("Prithvi-EO-2.0-300M-TL (TerraTorch), backbone frozen, UNet decoder, 6 Sentinel-2 bands, "
                    "trained on 26 of 252 training chips (frac10, stratified by event, subset seed 0), "
                    "252 samples per epoch, seed 1, 100 epochs, best validation epoch; "
                    "re-evaluated on CPU in float32 from the saved best checkpoint after a Drive sync loss"),
    "modality": "s2_6bands",
    "checkpoint_epoch": state["epoch"],
    "checkpoint_valid_iou": state["valid_iou"],
    "created": datetime.now().isoformat(timespec="seconds"),
    "code_commit": commit,
    "splits": {},
}
bolivia_predictions = {}
t0 = time.time()

with torch.no_grad():
    for split in EVAL_SPLITS:
        ds = PrithviSen1Floods11(DATA_ROOT, split, augment=False)
        split_total, event_totals, chips = {}, {}, {}
        for i, chip_id in enumerate(ds.chip_ids):
            image, _ = ds[i]
            pred = logits_of(model(image.unsqueeze(0))).argmax(dim=1)[0].numpy().astype("uint8")

            # Missing-input rule: no Sentinel-2 image gives "not water"
            chip = read_chip(DATA_ROOT, chip_id)
            pred[(chip["s2"] == 0).all(axis=0)] = 0

            counts = confusion_counts(pred, chip["label"])
            chips[chip_id] = counts
            add_counts(split_total, counts)
            add_counts(event_totals.setdefault(chip_id.split("_")[0], {}), counts)
            if split == "bolivia":
                bolivia_predictions[chip_id] = pred
            if (i + 1) % 10 == 0:
                print(f"{split}: {i + 1}/{len(ds)} chips, {time.time() - t0:.0f} s")

        results["splits"][split] = {
            "overall": {**split_total, **scores(split_total)},
            "events": {event: {**c, **scores(c)} for event, c in sorted(event_totals.items())},
            "chips": chips,
        }

(RESULTS_DIR / f"{METHOD}.json").write_text(json.dumps(results, indent=2))
np.savez_compressed(PRED_DIR / f"{METHOD}_bolivia.npz", **bolivia_predictions)
for split in ["test", "bolivia"]:
    s = results["splits"][split]["overall"]
    print(f"{METHOD} | {split}: IoU = {s['iou']:.3f}, F1 = {s['f1']:.3f}")

Best checkpoint: epoch 1, valid IoU 0.5169


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Prithvi_EO_V2_300M_TL.pt: reconstructing file:   0%|          |  0.00B / 1.33GB            

Prithvi_EO_V2_300M_TL.pt: downloading bytes:           |  0.00B            

valid: 10/89 chips, 164 s
valid: 20/89 chips, 316 s
valid: 30/89 chips, 468 s
valid: 40/89 chips, 623 s
valid: 50/89 chips, 776 s
valid: 60/89 chips, 928 s
valid: 70/89 chips, 1081 s
valid: 80/89 chips, 1236 s
test: 10/90 chips, 1528 s
test: 20/90 chips, 1679 s
test: 30/90 chips, 1837 s
test: 40/90 chips, 1990 s
test: 50/90 chips, 2143 s
test: 60/90 chips, 2295 s
test: 70/90 chips, 2449 s
test: 80/90 chips, 2601 s
test: 90/90 chips, 2753 s
bolivia: 10/15 chips, 2906 s
prithvi_frozen_frac10_seed1 | test: IoU = 0.552, F1 = 0.711
prithvi_frozen_frac10_seed1 | bolivia: IoU = 0.434, F1 = 0.605
